In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import 
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786269645.740232  456304 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786269645.855741  456304 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786269648.537157  456304 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
df = pd.read_csv("../Dataset/df_for_EDA.csv")

In [4]:
df = df.sort_index()

print(df.index.min())
print(df.index.max())
print(df.shape)

0
145197
(145198, 15)


In [5]:
test_size = int(len(df) * 0.20)

test_df = df.iloc[-test_size:].copy()

remaining_df = df.iloc[:-test_size].copy()

In [6]:
validation_hours = 60 * 24

val_df = remaining_df.iloc[-validation_hours:].copy()

train_df = remaining_df.iloc[:-validation_hours].copy()

In [7]:
print("Train:")
print(train_df.index.min(), "→", train_df.index.max())
print(train_df.shape)

print("\nValidation:")
print(val_df.index.min(), "→", val_df.index.max())
print(val_df.shape)

print("\nTest:")
print(test_df.index.min(), "→", test_df.index.max())
print(test_df.shape)

Train:
0 → 114718
(114719, 15)

Validation:
114719 → 116158
(1440, 15)

Test:
116159 → 145197
(29039, 15)


In [8]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform(
    train_df[["PJME_MW"]]
)


val_scaled = scaler.transform(
    val_df[["PJME_MW"]]
)


test_scaled = scaler.transform(
    test_df[["PJME_MW"]]
)

In [9]:
train_scaled

array([[-0.78296491],
       [-1.0265028 ],
       [-1.14067088],
       ...,
       [-0.34940542],
       [-0.60736944],
       [-0.85974915]], shape=(114719, 1))

In [10]:
test_scaled

array([[-1.08979162],
       [-1.40328848],
       [-1.55592624],
       ...,
       [ 1.57314781],
       [ 1.22598998],
       [ 0.98462377]], shape=(29039, 1))

In [12]:
val_scaled

array([[-1.08296636],
       [-1.20101243],
       [-1.27128164],
       ...,
       [ 0.05793884],
       [-0.28720243],
       [-0.71502249]], shape=(1440, 1))

In [13]:
def create_sequences(data, sequence_length, forecast_horizon):

    X = []
    y = []

    for i in range(
        sequence_length,
        len(data) - forecast_horizon + 1
    ):

        X.append(
            data[i-sequence_length:i]
        )

        y.append(
            data[i:i+forecast_horizon]
        )

    return np.array(X), np.array(y)

In [14]:
SEQUENCE_LENGTH = 24
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [15]:
X_train

array([[[-0.78296491],
        [-1.0265028 ],
        [-1.14067088],
        ...,
        [-0.0252053 ],
        [-0.17691234],
        [-0.42541406]],

       [[-1.0265028 ],
        [-1.14067088],
        [-1.16952314],
        ...,
        [-0.17691234],
        [-0.42541406],
        [-1.0548897 ]],

       [[-1.14067088],
        [-1.16952314],
        [-1.10390751],
        ...,
        [-0.42541406],
        [-1.0548897 ],
        [-1.22505598]],

       ...,

       [[-0.28410004],
        [-0.57572502],
        [-1.03069103],
        ...,
        [ 0.65468423],
        [ 0.55059892],
        [ 0.30799176]],

       [[-0.57572502],
        [-1.03069103],
        [-1.1124391 ],
        ...,
        [ 0.55059892],
        [ 0.30799176],
        [-0.05715996]],

       [[-1.03069103],
        [-1.1124391 ],
        [-1.14547959],
        ...,
        [ 0.30799176],
        [-0.05715996],
        [-0.39671692]]], shape=(114672, 24, 1))

In [16]:
y_train

array([[[-1.0548897 ],
        [-1.22505598],
        [-1.28369122],
        ...,
        [ 0.34087713],
        [ 0.02815586],
        [-0.37577577]],

       [[-1.22505598],
        [-1.28369122],
        [-1.27515963],
        ...,
        [ 0.02815586],
        [-0.37577577],
        [-0.79397841]],

       [[-1.28369122],
        [-1.27515963],
        [-1.18674142],
        ...,
        [-0.37577577],
        [-0.79397841],
        [-1.01828145]],

       ...,

       [[-0.05715996],
        [-0.39671692],
        [-1.38575996],
        ...,
        [-0.06569154],
        [-0.15659167],
        [-0.34940542]],

       [[-0.39671692],
        [-1.38575996],
        [-1.52847006],
        ...,
        [-0.15659167],
        [-0.34940542],
        [-0.60736944]],

       [[-1.38575996],
        [-1.52847006],
        [-1.61114885],
        ...,
        [-0.34940542],
        [-0.60736944],
        [-0.85974915]]], shape=(114672, 24, 1))

In [17]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (114672, 24, 1)
y_train shape: (114672, 24, 1)


In [18]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [19]:
val_input

array([[-1.38575996],
       [-1.52847006],
       [-1.61114885],
       ...,
       [ 0.05793884],
       [-0.28720243],
       [-0.71502249]], shape=(1464, 1))

In [20]:
X_val

array([[[-1.38575996],
        [-1.52847006],
        [-1.61114885],
        ...,
        [-0.34940542],
        [-0.60736944],
        [-0.85974915]],

       [[-1.52847006],
        [-1.61114885],
        [-1.63643335],
        ...,
        [-0.60736944],
        [-0.85974915],
        [-1.08296636]],

       [[-1.61114885],
        [-1.63643335],
        [-1.60742597],
        ...,
        [-0.85974915],
        [-1.08296636],
        [-1.20101243]],

       ...,

       [[-0.53244664],
        [-0.83694656],
        [-1.16083643],
        ...,
        [ 0.14573658],
        [ 0.24640925],
        [ 0.12339934]],

       [[-0.83694656],
        [-1.16083643],
        [-1.39010331],
        ...,
        [ 0.24640925],
        [ 0.12339934],
        [-0.19366527]],

       [[-1.16083643],
        [-1.39010331],
        [-1.55933888],
        ...,
        [ 0.12339934],
        [-0.19366527],
        [-0.6070592 ]]], shape=(1417, 24, 1))

In [21]:
y_val

array([[[-1.08296636],
        [-1.20101243],
        [-1.27128164],
        ...,
        [-0.68663559],
        [-0.91869462],
        [-1.16223251]],

       [[-1.20101243],
        [-1.27128164],
        [-1.28803457],
        ...,
        [-0.91869462],
        [-1.16223251],
        [-1.44020697]],

       [[-1.27128164],
        [-1.28803457],
        [-1.23560412],
        ...,
        [-1.16223251],
        [-1.44020697],
        [-1.58896673]],

       ...,

       [[-0.19366527],
        [-0.6070592 ],
        [-0.99253159],
        ...,
        [ 0.13953179],
        [ 0.17862195],
        [ 0.05793884]],

       [[-0.6070592 ],
        [-0.99253159],
        [-1.37412598],
        ...,
        [ 0.17862195],
        [ 0.05793884],
        [-0.28720243]],

       [[-0.99253159],
        [-1.37412598],
        [-1.5397938 ],
        ...,
        [ 0.05793884],
        [-0.28720243],
        [-0.71502249]]], shape=(1417, 24, 1))

In [22]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [23]:
test_input

array([[-0.99253159],
       [-1.37412598],
       [-1.5397938 ],
       ...,
       [ 1.57314781],
       [ 1.22598998],
       [ 0.98462377]], shape=(29063, 1))

In [24]:
X_test

array([[[-0.99253159],
        [-1.37412598],
        [-1.5397938 ],
        ...,
        [ 0.05793884],
        [-0.28720243],
        [-0.71502249]],

       [[-1.37412598],
        [-1.5397938 ],
        [-1.63333096],
        ...,
        [-0.28720243],
        [-0.71502249],
        [-1.08979162]],

       [[-1.5397938 ],
        [-1.63333096],
        [-1.67878102],
        ...,
        [-0.71502249],
        [-1.08979162],
        [-1.40328848]],

       ...,

       [[ 0.92242078],
        [ 0.59372219],
        [ 0.84051759],
        ...,
        [ 2.09822791],
        [ 2.00856874],
        [ 1.76875372]],

       [[ 0.59372219],
        [ 0.84051759],
        [ 0.76001048],
        ...,
        [ 2.00856874],
        [ 1.76875372],
        [ 1.44579456]],

       [[ 0.84051759],
        [ 0.76001048],
        [ 0.74496388],
        ...,
        [ 1.76875372],
        [ 1.44579456],
        [ 1.11538966]]], shape=(29016, 24, 1))

In [25]:
y_test

array([[[-1.08979162],
        [-1.40328848],
        [-1.55592624],
        ...,
        [-0.06398522],
        [-0.34754399],
        [-0.73751484]],

       [[-1.40328848],
        [-1.55592624],
        [-1.64729173],
        ...,
        [-0.34754399],
        [-0.73751484],
        [-1.09801297]],

       [[-1.55592624],
        [-1.64729173],
        [-1.67986686],
        ...,
        [-0.73751484],
        [-1.09801297],
        [-1.39692858]],

       ...,

       [[ 1.44579456],
        [ 1.11538966],
        [ 1.03379671],
        ...,
        [ 1.86508304],
        [ 1.78240425],
        [ 1.57314781]],

       [[ 1.11538966],
        [ 1.03379671],
        [ 0.93653667],
        ...,
        [ 1.78240425],
        [ 1.57314781],
        [ 1.22598998]],

       [[ 1.03379671],
        [ 0.93653667],
        [ 0.86052803],
        ...,
        [ 1.57314781],
        [ 1.22598998],
        [ 0.98462377]]], shape=(29016, 24, 1))

In [26]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (114672, 24, 1)
y_train: (114672, 24, 1)
X_val: (1417, 24, 1)
y_val: (1417, 24, 1)
X_test: (29016, 24, 1)
y_test: (29016, 24, 1)


# Basic RNN

In [27]:
def build_rnn_model(sequence_length):

    model = Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        SimpleRNN(
            64,
            activation="tanh"
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [28]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

E0000 00:00:1786271154.853900  456304 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [29]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 0.1592 - mae: 0.2898 - val_loss: 0.0939 - val_mae: 0.2302
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1164 - mae: 0.2507 - val_loss: 0.0655 - val_mae: 0.1853
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1106 - mae: 0.2443 - val_loss: 0.0778 - val_mae: 0.2062
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1059 - mae: 0.2391 - val_loss: 0.0731 - val_mae: 0.1988
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1027 - mae: 0.2354 - val_loss: 0.0693 - val_mae: 0.1915
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0996 - mae: 0.2317 - val_loss: 0.0800 - val_mae: 0.2085
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0973 - mae: 0.2290 - val_loss: 0.0705 - val_mae: 0.1926
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0951 - mae: 0.2265 - val_loss: 0.0551 - val_mae: 0.1710
Epoch 9/10
1792/1792 ━━━━━━━━

In [31]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0840 - mae: 0.2117 - val_loss: 0.0637 - val_mae: 0.1853 - learning_rate: 0.0010
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.0839 - mae: 0.2118 - val_loss: 0.0491 - val_mae: 0.1619 - learning_rate: 0.0010
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0829 - mae: 0.2105 - val_loss: 0.0617 - val_mae: 0.1848 - learning_rate: 0.0010
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0823 - mae: 0.2099 - val_loss: 0.0478 - val_mae: 0.1597 - learning_rate: 0.0010
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0812 - mae: 0.2084 - val_loss: 0.0513 - val_mae: 0.1667 - learning_rate: 0.0010
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0804 - mae: 0.2072 - val_loss: 0.0471 - val_mae: 0.1594 - learning_rate: 0.0010
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 0.0799 - mae: 0.2065 - val_loss: 0.0464 - val_mae: 0.1602 - learnin

# LSTM

In [32]:
def build_lstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        LSTM(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [33]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - loss: 0.1766 - mae: 0.3088 - val_loss: 0.0767 - val_mae: 0.2018
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 0.1092 - mae: 0.2440 - val_loss: 0.0606 - val_mae: 0.1772
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.1008 - mae: 0.2337 - val_loss: 0.0657 - val_mae: 0.1883
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0969 - mae: 0.2283 - val_loss: 0.0604 - val_mae: 0.1767
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0939 - mae: 0.2242 - val_loss: 0.0562 - val_mae: 0.1718
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0914 - mae: 0.2207 - val_loss: 0.0549 - val_mae: 0.1687
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.0891 - mae: 0.2175 - val_loss: 0.0542 - val_mae: 0.1693
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0876 - mae: 0.2154 - val_loss: 0.0586 - val_mae: 0.1754
Epoch 9/10
1792/1792 ━━━

# GRU

In [34]:
def build_gru_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        GRU(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [35]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 33s 16ms/step - loss: 0.1789 - mae: 0.3077 - val_loss: 0.0730 - val_mae: 0.1962
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1117 - mae: 0.2462 - val_loss: 0.0811 - val_mae: 0.2079
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1028 - mae: 0.2355 - val_loss: 0.0894 - val_mae: 0.2232
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0978 - mae: 0.2292 - val_loss: 0.0681 - val_mae: 0.1905
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0942 - mae: 0.2245 - val_loss: 0.0704 - val_mae: 0.1943
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 0.0912 - mae: 0.2206 - val_loss: 0.0607 - val_mae: 0.1807
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0884 - mae: 0.2168 - val_loss: 0.0534 - val_mae: 0.1670
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0859 - mae: 0.2136 - val_loss: 0.0476 - val_mae: 0.1569
Epoch 9/10
1792/1792 ━━━

# LSTM + Bidirectional

In [ ]:
def build_bilstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ), 33184.0
22364     33184.0
11043     33184.0
7284      33184.0
32546     33184.0
           ...   
4971      33184.0

        Bidirectional(
            LSTM(64)
            
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model




In [38]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 75s 37ms/step - loss: 0.1511 - mae: 0.2835 - val_loss: 0.0674 - val_mae: 0.1892
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.1049 - mae: 0.2387 - val_loss: 0.0759 - val_mae: 0.2039
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0978 - mae: 0.2294 - val_loss: 0.0597 - val_mae: 0.1741
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0933 - mae: 0.2233 - val_loss: 0.0579 - val_mae: 0.1750
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 24ms/step - loss: 0.0899 - mae: 0.2187 - val_loss: 0.0630 - val_mae: 0.1819
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - loss: 0.0867 - mae: 0.2142 - val_loss: 0.0603 - val_mae: 0.1767
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 25ms/step - loss: 0.0837 - mae: 0.2101 - val_loss: 0.0546 - val_mae: 0.1680
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 47s 26ms/step - loss: 0.0809 - mae: 0.2062 - val_loss: 0.0502 - val_mae: 0.1590
Epoch 9/10
1792/1792 ━━━

In [39]:
rnn_pred = rnn_model.predict(
    X_test
)

lstm_pred = lstm_model.predict(
    X_test
)

gru_pred = gru_model.predict(
    X_test
)

bilstm_pred = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step


In [40]:
rnn_pred

array([[-0.98236305, -1.1773653 , -1.3379735 , ..., -0.05086365,
        -0.3281464 , -0.6497523 ],
       [-1.2789485 , -1.3627292 , -1.4237107 , ..., -0.3937418 ,
        -0.7459058 , -0.947224  ],
       [-1.5271153 , -1.5898683 , -1.6117347 , ..., -0.70814604,
        -1.0258298 , -1.2596714 ],
       ...,
       [ 1.2444727 ,  0.8547008 ,  0.584122  , ...,  2.1134279 ,
         2.014485  ,  1.8637037 ],
       [ 0.8579916 ,  0.50949216,  0.3693345 , ...,  2.0326931 ,
         1.8534814 ,  1.5468844 ],
       [ 0.5841448 ,  0.32907543,  0.2987256 , ...,  1.9806972 ,
         1.6389323 ,  1.2001114 ]], shape=(29016, 24), dtype=float32)

In [41]:
lstm_pred

array([[-1.0339231 , -1.2772336 , -1.436019  , ..., -0.04974754,
        -0.3624664 , -0.6767298 ],
       [-1.2767748 , -1.4241961 , -1.5378155 , ..., -0.34222573,
        -0.7226193 , -0.99429274],
       [-1.5397735 , -1.6436062 , -1.6772805 , ..., -0.73920304,
        -1.0781935 , -1.2977225 ],
       ...,
       [ 1.2708527 ,  0.79982126,  0.47288364, ...,  2.084054  ,
         1.9476794 ,  1.6866069 ],
       [ 0.86661685,  0.5300539 ,  0.32884455, ...,  2.0595398 ,
         1.7841105 ,  1.4589174 ],
       [ 0.58009523,  0.3989885 ,  0.32478458, ...,  1.8437864 ,
         1.4647771 ,  1.1074753 ]], shape=(29016, 24), dtype=float32)

In [42]:
gru_pred

array([[-1.0418394 , -1.240251  , -1.3790203 , ...,  0.04230755,
        -0.25595686, -0.6157472 ],
       [-1.2761693 , -1.4134693 , -1.5176489 , ..., -0.2962089 ,
        -0.66202915, -0.9760723 ],
       [-1.4770218 , -1.5842029 , -1.6263608 , ..., -0.65823865,
        -1.0153581 , -1.2890549 ],
       ...,
       [ 1.4874526 ,  1.1683657 ,  0.8905362 , ...,  2.2833655 ,
         2.1807811 ,  1.9652877 ],
       [ 1.3095893 ,  1.0279827 ,  0.77259773, ...,  2.100257  ,
         1.9594995 ,  1.6773474 ],
       [ 1.1330478 ,  0.8729358 ,  0.7200676 , ...,  1.8706336 ,
         1.5879427 ,  1.2296402 ]], shape=(29016, 24), dtype=float32)

In [43]:
bilstm_pred

array([[-1.0080543 , -1.1750776 , -1.307158  , ..., -0.03861562,
        -0.37374806, -0.7252225 ],
       [-1.2875327 , -1.3820951 , -1.4739033 , ..., -0.4009152 ,
        -0.68230283, -0.8928752 ],
       [-1.5442355 , -1.6039561 , -1.6016681 , ..., -0.86552817,
        -1.1030685 , -1.218047  ],
       ...,
       [ 1.414811  ,  0.9988717 ,  0.60385287, ...,  2.1057782 ,
         1.9298843 ,  1.7164263 ],
       [ 0.9204057 ,  0.60039264,  0.37882727, ...,  1.8782209 ,
         1.6182475 ,  1.2880441 ],
       [ 0.5800745 ,  0.45264882,  0.37300885, ...,  1.7966943 ,
         1.4428414 ,  0.9681557 ]], shape=(29016, 24), dtype=float32)

# ReScaling

In [45]:
rnn_pred_original = scaler.inverse_transform(
    rnn_pred.reshape(-1, 1)
).reshape(rnn_pred.shape)

lstm_pred_original = scaler.inverse_transform(
    lstm_pred.reshape(-1, 1)
).reshape(lstm_pred.shape)

gru_pred_original = scaler.inverse_transform(
    gru_pred.reshape(-1, 1)
).reshape(gru_pred.shape)

bilstm_pred_original = scaler.inverse_transform(
    bilstm_pred.reshape(-1, 1)
).reshape(bilstm_pred.shape)

In [46]:
rnn_pred_original

array([[25927.555, 24670.445, 23635.062, ..., 31932.59 , 30145.05 ,
        28071.773],
       [24015.574, 23475.473, 23082.346, ..., 29722.18 , 27451.906,
        26154.082],
       [22415.734, 22011.188, 21870.223, ..., 27695.33 , 25647.34 ,
        24139.848],
       ...,
       [40283.152, 37770.434, 36026.113, ..., 45884.992, 45247.14 ,
        44275.11 ],
       [37791.65 , 35545.   , 34641.453, ..., 45364.523, 44209.21 ,
        42232.69 ],
       [36026.258, 34381.918, 34186.266, ..., 45029.324, 42826.09 ,
        39997.17 ]], shape=(29016, 24), dtype=float32)

In [47]:
lstm_pred_original

array([[25595.164, 24026.63 , 23003.   , ..., 31939.785, 29923.8  ,
        27897.86 ],
       [24029.59 , 23079.217, 22346.754, ..., 30054.285, 27602.027,
        25850.646],
       [22334.133, 21664.76 , 21447.674, ..., 27495.117, 25309.77 ,
        23894.547],
       ...,
       [40453.215, 37416.65 , 35309.   , ..., 45695.625, 44816.47 ,
        43133.43 ],
       [37847.254, 35677.555, 34380.43 , ..., 45537.594, 43762.   ,
        41665.6  ],
       [36000.152, 34832.625, 34354.258, ..., 44146.71 , 41703.375,
        39399.98 ]], shape=(29016, 24), dtype=float32)

In [50]:
y_test_original = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

# Error Calculation

In [48]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [51]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original
)

In [52]:
deep_results

{'RNN': {'MAE': 1582.632554135508,
  'MSE': 4720266.125651733,
  'RMSE': np.float64(2172.6173445067893),
  'MAPE': 5.1174328903127275,
  'R2': 0.8892683575234632,
  'Bias': np.float64(321.9579765278455)},
 'LSTM': {'MAE': 1576.632464123938,
  'MSE': 4697914.9737056345,
  'RMSE': np.float64(2167.4674100677116),
  'MAPE': 5.072953668259601,
  'R2': 0.8897926880803325,
  'Bias': np.float64(137.18421656269655)},
 'GRU': {'MAE': 1554.157947776927,
  'MSE': 4582343.315316676,
  'RMSE': np.float64(2140.6408655626183),
  'MAPE': 5.0580674280512286,
  'R2': 0.8925038571577708,
  'Bias': np.float64(461.5139159595317)},
 'Bi-LSTM': {'MAE': 1480.638330575112,
  'MSE': 4262538.484320015,
  'RMSE': np.float64(2064.5916023078303),
  'MAPE': 4.787187228935753,
  'R2': 0.9000060854782773,
  'Bias': np.float64(122.36444579808877)}}

In [53]:
deep_results_df = pd.DataFrame(
    deep_results
).T

deep_results_df

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,1582.632554,4.720266e+06,2172.617345,5.117433,0.889268,321.957977
LSTM,1576.632464,4.697915e+06,2167.467410,5.072954,0.889793,137.184217
GRU,1554.157948,4.582343e+06,2140.640866,5.058067,0.892504,461.513916
Bi-LSTM,1480.638331,4.262538e+06,2064.591602,4.787187,0.900006,122.364446
